# PyTorch and BM25 development demo

This experiment joins a temporal sensor classifier with lexical evidence retrieval. The classifier uses 32 consecutive readings to score an annotated anomaly. BM25 ranks the project-authored analytical notes used by the investigation graph.

The model is a research artifact. It does not replace the active gradient-boosting detector. Training and threshold selection use the existing development allocation, so the results below do not establish performance on new equipment. Source anomaly annotations are not confirmed physical failures.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from forge.ml.torch_artifacts import load_torch_artifact
from IPython.display import Markdown, display

from forge.agents.service import investigate_recording, load_development_recording
from forge.rag.evaluate import evaluate_retrieval
from forge.reports.incident import markdown

pio.renderers.default = "plotly_mimetype"

root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src/forge").is_dir()
)
# Set RUN_ID to compare a specific saved experiment. None selects the newest local run.
RUN_ID = None
runs = sorted((root / "models").glob("pytorch-*"))
if RUN_ID is None and not runs:
    raise FileNotFoundError(
        "Train first with: python -m forge.ml.torch_training. "
        "Install requirements-torch.txt in the project environment if needed."
    )
artifact = root / "models" / RUN_ID if RUN_ID else runs[-1]
detector, metadata = load_torch_artifact(artifact, root=root)
display(
    pd.Series(
        {
            "run": metadata["run_id"],
            "torch": metadata["torch_version"],
            "threshold": metadata["threshold"],
            "window": metadata["config"]["window"],
        }
    )
)

## Training and selection

The frozen configuration uses a small temporal convolutional network and eight CPU epochs. Normalization is fitted on training data. Windows stay within their source recording and restart after a sampling gap. Labels and recording identifiers are excluded from the model inputs.

The threshold is selected on validation data with a minimum recall requirement. The following metrics describe that same development selection, rather than an independent test.

In [ ]:
history = pd.DataFrame(json.loads((artifact / "history.json").read_text()))
fig = go.Figure(go.Scatter(x=history["epoch"], y=history["training_loss"], mode="lines+markers"))
fig.update_layout(title="Training loss", xaxis_title="Epoch", yaxis_title="Binary cross-entropy")
fig.show()
display(pd.Series(metadata["threshold_selection"], name="Selection policy"))

In [ ]:
metrics = metadata["validation_metrics"]
display(
    pd.Series(
        {
            key: metrics[key]
            for key in ("precision", "recall", "fpr_available_normal", "unavailable_rows")
        },
        name="Development validation",
    )
)
confusion = pd.DataFrame(
    [[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]],
    index=["Annotated normal", "Annotated anomaly"],
    columns=["No alert", "Alert"],
)
display(confusion)

## One recording

The trace shows how scores develop over time. Unavailable startup readings are left blank. The source annotations are overlaid for inspection and are not passed into the network. Scores are uncalibrated model outputs, not failure probabilities.

In [ ]:
recording = "valve1/1"
frame, record = load_development_recording(recording, root=root)
scores = detector.score(frame)
ready = detector.readiness(frame)
fig = go.Figure()
fig.add_scatter(x=frame["datetime"], y=np.where(ready, scores, np.nan), name="Model score")
fig.add_scatter(
    x=frame["datetime"],
    y=frame["anomaly"],
    name="Source anomaly annotation",
    mode="lines",
    line={"dash": "dot"},
)
fig.add_hline(y=metadata["threshold"], line_dash="dash", annotation_text="Selected threshold")
fig.update_layout(
    title=f"Research detector: {recording}",
    xaxis_title="Recorded time",
    yaxis_title="Score / source annotation",
)
fig.show()

## Retrieval comparison

Both methods search the same seven versioned notes. The nine-query fixture was authored with this corpus and includes two unrelated queries. It checks known retrieval behavior. A perfect result on this small fixture cannot establish broad retrieval quality, and the two methods have different score scales.

In [ ]:
retrieval = {backend: evaluate_retrieval(root, backend=backend) for backend in ("tfidf", "bm25")}
display(
    pd.DataFrame(
        {
            backend: {
                key: result[key]
                for key in (
                    "recall_at_3",
                    "mrr",
                    "correct_abstentions",
                    "expected_abstentions",
                    "retrieved_citations",
                    "verified_citations",
                )
            }
            for backend, result in retrieval.items()
        }
    ).T
)
display(
    pd.DataFrame(retrieval["bm25"]["results"])[
        ["query", "expected", "retrieved", "recall_at_3", "correct_abstention"]
    ]
)

## Combined investigation

The selected PyTorch checkpoint supplies the scores and BM25 supplies cited passages. The graph verifies each citation before proposing analytical checks. A report remains unreviewed until a person reviews it. No root cause or equipment action is inferred from the score alone.

In [ ]:
report = investigate_recording(
    recording, root=root, retrieval_backend="bm25", torch_artifact=artifact
)
display(Markdown(markdown(report)))